# Phase 2a : Calcul des poids Grid Search

In [ ]:
model_names_t2i = ["siglip", "blip", "convnext_clip", "eva_clip"]
model_names_i2t = ['siglip', 'blip', 'convnext_clip', 'eva_clip']

In [ ]:
import numpy as np
from collections import defaultdict
import json
import config

from utils_indexation import SearchIndex
from utils_evaluation import GridSearchOptimizer

/home/marion/MIRAGE_TFE/env_marion/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [4]:
prefix_val = "val_"

all_model_names = list(set(model_names_t2i + model_names_i2t))

print(f"Modèles T2I : {model_names_t2i}")
print(f"Modèles I2T : {model_names_i2t}")

Modèles T2I : ['siglip', 'blip', 'convnext_clip']
Modèles I2T : ['siglip', 'blip', 'convnext_clip', 'eva_clip']


## Rechargement des Matrices (Validation) et des IDs

In [ ]:
print("\nChargement des données de Validation...")
txt_vecs, img_vecs = {}, {}

for name in all_model_names:
    img_vecs[name] = np.load(f"{config.INDEX_DIR}/{prefix_val}{name}_img_vecs.npy")
    txt_vecs[name] = np.load(f"{config.INDEX_DIR}/{prefix_val}{name}_txt_vecs.npy")

# On charge les IDs une seule fois via le premier modèle (les IDs sont identiques pour tous)
modele_ref = all_model_names[0]
idx_img = SearchIndex(dimension=0); idx_img.load_from_disk(f"{config.INDEX_DIR}/{prefix_val}{modele_ref}_img")
img_ids = idx_img.image_ids

idx_txt = SearchIndex(dimension=0); idx_txt.load_from_disk(f"{config.INDEX_DIR}/{prefix_val}{modele_ref}_txt")
txt_to_img_id = idx_txt.image_ids

# Calcul de toutes les matrices de base
similarity_matrices = {name: txt_vecs[name] @ img_vecs[name].T for name in all_model_names}

img_id_to_idx = {img_id: idx for idx, img_id in enumerate(img_ids)}
targets_t2i = np.array([img_id_to_idx[iid] for iid in txt_to_img_id])

img_to_txt_indices = defaultdict(list)
for txt_idx, iid in enumerate(txt_to_img_id):
    img_to_txt_indices[img_id_to_idx[iid]].append(txt_idx)
targets_i2t = [img_to_txt_indices[i] for i in range(len(img_id_to_idx))]

print(f"Targets Validation prêtes : T2I ({len(targets_t2i)}) | I2T ({len(targets_i2t)})")


Chargement des données de Validation...
  Chargé : ./data/flickr30k/index_sauvegardes/val_eva_clip_img_index.bin (1014 vecteurs)
  Chargé : ./data/flickr30k/index_sauvegardes/val_eva_clip_txt_index.bin (5070 vecteurs)
Targets Validation prêtes : T2I (5070) | I2T (1014)


## Lancement du Grid Search

In [6]:
sim_matrices_t2i = {name: similarity_matrices[name] for name in model_names_t2i}
sim_matrices_i2t = {name: similarity_matrices[name] for name in model_names_i2t}

print("\n--- Lancement du Grid Search T2I ---")
opt_t2i = GridSearchOptimizer(sim_matrices_t2i, targets_t2i, targets_i2t, step=0.1)
df_recap_t2i = opt_t2i.optimize(task="t2i")

print("\n--- Lancement du Grid Search I2T ---")
opt_i2t = GridSearchOptimizer(sim_matrices_i2t, targets_t2i, targets_i2t, step=0.1)
df_recap_i2t = opt_i2t.optimize(task="i2t")

metrics_list = ['R@1', 'R@5', 'R@10', 'mAP', 'NDCG']

def get_best_weights(df, metric, task_models):
    """
    Assigne le poids trouvé si le modèle est dans task_models, sinon 0.0.
    """
    return {name: (df.loc[df[metric].idxmax()][name] if name in task_models else 0.0) for name in all_model_names}

BEST_WEIGHTS = {'t2i': {}, 'i2t': {}}
for m in metrics_list:
    BEST_WEIGHTS['t2i'][m] = get_best_weights(df_recap_t2i, metric=m, task_models=model_names_t2i)
    BEST_WEIGHTS['i2t'][m] = get_best_weights(df_recap_i2t, metric=m, task_models=model_names_i2t)


--- Lancement du Grid Search T2I ---
🚀 GridSearch initialisé sur : cuda

⚡ Optimisation pour la tâche : T2I...


100%|██████████| 1331/1331 [00:44<00:00, 29.64it/s]



--- Meilleures combinaisons (T2I) ---
Métrique                                 Meilleurs Poids  Score Maximal
     R@1 58.3% siglip + 25.0% blip + 16.7% convnext_clip         0.8661
     R@5 66.7% siglip + 22.2% blip + 11.1% convnext_clip         0.9708
    R@10  58.3% siglip + 41.7% blip + 0.0% convnext_clip         0.9868
     mAP  61.5% siglip + 30.8% blip + 7.7% convnext_clip         0.9116
    NDCG  61.5% siglip + 30.8% blip + 7.7% convnext_clip         0.9326

--- Lancement du Grid Search I2T ---
🚀 GridSearch initialisé sur : cuda

⚡ Optimisation pour la tâche : I2T...


100%|██████████| 14641/14641 [02:25<00:00, 100.44it/s]


--- Meilleures combinaisons (I2T) ---
Métrique                                                  Meilleurs Poids  Score Maximal
     R@1 29.6% siglip + 25.9% blip + 33.3% convnext_clip + 11.1% eva_clip         0.9635
     R@5  41.2% siglip + 17.6% blip + 5.9% convnext_clip + 35.3% eva_clip         1.0000
    R@10  0.0% siglip + 33.3% blip + 33.3% convnext_clip + 33.3% eva_clip         1.0000
     mAP  46.7% siglip + 26.7% blip + 6.7% convnext_clip + 20.0% eva_clip         0.8842
    NDCG 41.2% siglip + 23.5% blip + 11.8% convnext_clip + 23.5% eva_clip         0.9476


In [ ]:
with open(config.BEST_WEIGHTS_FILE, 'w') as f: json.dump(BEST_WEIGHTS, f, indent=4)
print(f"Meilleurs poids trouvés et sauvegardés dans : {config.BEST_WEIGHTS_FILE}")

Meilleurs poids trouvés et sauvegardés dans : ./data/flickr30k/grid_search/best_weights.json
